In [23]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [24]:
from sklearn.datasets import fetch_openml

a = fetch_openml(
    "Fashion-MNIST",
    version=1,
    as_frame=False,
    parser="liac-arff"
)

x = a.data
y = a.target.astype(int)

print("Dataset shape:", x.shape)
print("Labels shape:", y.shape)

Dataset shape: (70000, 784)
Labels shape: (70000,)


In [25]:
df=pd.DataFrame(x)
df["Labels"]=y
df.head()

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,Labels
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,114.0,130.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.0,96.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [26]:
print("Total datasetsize: ",x.shape[0])
print("Pixels per image: ",x.shape[1])


Total datasetsize:  70000
Pixels per image:  784


In [27]:
#Select 12000 images
x,_,y,_=train_test_split(
    x,y,stratify=y,random_state=42,train_size=12000
)

In [28]:
print("selected images: ",len(x))

selected images:  12000


In [29]:
#split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=2000,
    stratify=y, random_state=42
)

In [30]:
print("Training images:", len(x_train))
print("Testing images:", len(x_test))

Training images: 10000
Testing images: 2000


In [31]:
#Normalization
print("Before Scaling:",x_train.min(),"-",x_train.max())
x_train=x_train/255.0
x_test=x_test/255.0
print("After Scaling:",x_train.min(),"-",x_train.max())

#test different k values

r = {}

for i in [1, 3, 5, 7, 9, 15]:
    m = KNeighborsClassifier(n_neighbors=i)
    m.fit(x_train, y_train)

    start = time.time()
    p = m.predict(x_test)
    elapsed = time.time() - start

    acc = accuracy_score(y_test, p)

    # Store accuracy, time, and predictions in this order
    r[i] = [acc, elapsed, p]

    print("K =", i, "| Accuracy =", round(acc * 100, 2),
          "% | Time =", round(elapsed, 2), "seconds")


Before Scaling: 0.0 - 255.0
After Scaling: 0.0 - 1.0
K = 1 | Accuracy = 79.3 % | Time = 0.31 seconds
K = 3 | Accuracy = 80.9 % | Time = 0.31 seconds
K = 5 | Accuracy = 81.0 % | Time = 0.29 seconds
K = 7 | Accuracy = 81.1 % | Time = 0.36 seconds
K = 9 | Accuracy = 80.95 % | Time = 0.31 seconds
K = 15 | Accuracy = 80.25 % | Time = 0.27 seconds


In [32]:
best = 1

for i in r:
    if r[i][0] > r[best][0]:
        best = i

best_predictions = r[best][2]

print("Best K:", best)

Best K: 7


In [33]:
print(classification_report(
    y_test, best_predictions,
    labels=list(range(10)),
    target_names=names
))

              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000



In [34]:
#  Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, r[best][2]))


Confusion Matrix:
[[166   0   2   5   1   0  25   0   1   0]
 [  1 188   6   5   0   0   0   0   0   0]
 [  3   0 146   2  25   0  24   0   0   0]
 [ 15   4   2 166   7   0   6   0   0   0]
 [  1   1  27  13 138   0  20   0   0   0]
 [  1   0   1   0   0 151   1  29   1  16]
 [ 38   0  31   4  15   0 109   0   3   0]
 [  0   0   0   0   0   0   0 184   0  16]
 [  1   0   4   1   0   0   4   3 187   0]
 [  1   0   0   0   0   0   0  12   0 187]]
